# 04. 딥러닝 모델 학습 (PyTorch)

**목표:** Tabular Neural Network를 학습하고 ML 모델과 성능을 비교합니다.

구조: `입력 → [Linear→BatchNorm→ReLU→Dropout] × 3 → 출력`

- Early Stopping (patience=15)
- ReduceLROnPlateau 스케줄러
- CUDA/CPU 자동 선택

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

from teen_mind.data.loader import load_raw_data
from teen_mind.data.preprocessor import MentalHealthPreprocessor
from teen_mind.models.dl_classifier import DLClassifier
from teen_mind.utils.visualization import plot_training_history

In [ ]:
import os
raw_files = [f for f in os.listdir('../data/raw') if f.endswith('.csv')]
df = load_raw_data(raw_files[0])

preprocessor = MentalHealthPreprocessor()
X_train, X_test, y_train, y_test = preprocessor.fit_transform(df)

input_dim = X_train.shape[1]
print(f'입력 피처 수: {input_dim}')

## 1. 모델 학습

In [ ]:
dl_clf = DLClassifier(
    input_dim=input_dim,
    output_dim=2,
    hidden_dims=[128, 64, 32],
    dropout_rate=0.3,
    learning_rate=1e-3,
    batch_size=64,
    epochs=100,
    patience=15,
    pos_weight=37.0,
)

history = dl_clf.fit(X_train, y_train, X_val=X_test, y_val=y_test)

## 2. 학습 곡선

In [ ]:
fig = plot_training_history(history)
fig.show()

## 3. 테스트 성능 평가

In [ ]:
metrics = dl_clf.evaluate(X_test, y_test)
print('DL 모델 성능:', metrics)

## 4. ML vs DL 비교

In [ ]:
import pandas as pd

ml_results = pd.read_csv('../models/saved/model_comparison.csv') if os.path.exists('../models/saved/model_comparison.csv') else None

dl_row = pd.DataFrame([{
    'Model': 'Neural Network (PyTorch)',
    'Accuracy': metrics['accuracy'],
    'F1 (binary)': metrics['f1_binary'],
    'AUC-ROC': metrics['auc_roc'],
}])

if ml_results is not None:
    comparison = pd.concat([ml_results, dl_row], ignore_index=True)
    print(comparison)
else:
    print(dl_row)

## 5. 모델 저장

In [ ]:
dl_clf.save('../models/saved/dl_model.pt')
print('DL 모델 저장 완료!')